In [1]:
import pandas as pd

In [2]:
!pip install -q sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer,util

In [5]:
df=pd.read_csv("/content/clean_jobs_descriptions_combined (1).csv")


In [6]:
print(df.shape)


(8785, 13)


In [7]:
#load the embedding model    #creates text into numbers that represent meaning
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
#test semantic similarity
word1 = "ML"
word2 = "Machine Learning"

embedding1 = model.encode(word1, convert_to_tensor=True)
embedding2 = model.encode(word2, convert_to_tensor=True)

similarity = util.cos_sim(embedding1, embedding2)

print("Similarity:", similarity.item())


Similarity: 0.37266361713409424


In [9]:
#compare a job description with skills
job_description = df["Job Description"].iloc[0]

print(job_description[:500])

Social Media Managers oversee an organizations social media presence. They create and schedule content, engage with followers, and analyze social media metrics to drive brand awareness and engagement.


In [10]:
#creating embeddings
job_embedding = model.encode(
    job_description,
    convert_to_tensor=True
)

In [11]:
skills = pd.DataFrame({
    'skill': [
        'Python', 'Java', 'SQL', 'Data Analysis', 'Machine Learning', 'Deep Learning', 'Natural Language Processing',
        'Computer Vision', 'Web Development', 'Frontend', 'Backend', 'Fullstack', 'JavaScript', 'HTML', 'CSS',
        'React', 'Angular', 'Vue.js', 'Node.js', 'Express.js', 'MongoDB', 'PostgreSQL', 'MySQL', 'AWS', 'Azure',
        'Google Cloud Platform', 'Docker', 'Kubernetes', 'CI/CD', 'Git', 'Agile', 'Scrum', 'Project Management',
        'Communication', 'Teamwork', 'Problem Solving', 'Critical Thinking', 'Leadership', 'Time Management',
        'Statistical Analysis', 'Big Data', 'Spark', 'Hadoop', 'Tableau', 'Power BI', 'Excel', 'Data Visualization',
        'Cloud Computing', 'Cybersecurity', 'Networking', 'Operating Systems', 'System Design', 'Algorithms',
        'Data Structures', 'Testing', 'DevOps', 'UI/UX Design', 'Copywriting', 'Content Creation', 'SEO',
        'SEM', 'Social Media Marketing', 'Email Marketing', 'Marketing Strategy', 'Sales', 'Customer Service',
        'Financial Modeling', 'Budgeting', 'Auditing', 'Compliance', 'Risk Management', 'Business Analysis',
        'Strategic Planning', 'Public Speaking', 'Mentoring', 'Coaching', 'Research', 'Technical Writing',
        'Software Development', 'Mobile Development', 'iOS Development', 'Android Development', 'Game Development',
        'AR/VR', 'Blockchain', 'Quantum Computing', 'Embedded Systems', 'Robotics'
    ]
})
display(skills.head())

,skill
0,Python
1,Java
2,SQL
3,Data Analysis
4,Machine Learning


In [12]:
# The `skills` DataFrame is now defined. Correcting the typo in skill embeddings creation.
skill_embeddings = model.encode(
  skills["skill"].tolist(),
    convert_to_tensor=True
)

In [13]:
#calculate similarity
similarities = util.cos_sim(
    job_embedding,
    skill_embeddings
)[0]

In [14]:
#find most similar skills
top_indices = similarities.argsort(descending=True)[:10]

for i in top_indices:
    print(
        skills.iloc[i.item()]["skill"],
        "→",
        round(similarities[i].item(), 3)
    )

Social Media Marketing → 0.633
Email Marketing → 0.29
Leadership → 0.283
Marketing Strategy → 0.271
Time Management → 0.256
Project Management → 0.241
SEM → 0.228
SEO → 0.222
Content Creation → 0.21
Communication → 0.199


In [15]:
#create semantic skill extraction
def extract_semantic_skills(job_description, threshold=0.25): # Lowered the default threshold

    job_embedding = model.encode(
        str(job_description),
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        job_embedding,
        skill_embeddings
    )[0]

    matched_skills = []

    for i, score in enumerate(similarities):
        if score.item() >= threshold:
            matched_skills.append(skills.iloc[i]["skill"])

    return ", ".join(matched_skills)

In [16]:
df["semantic_skills"] = df["Job Description"].apply(
    extract_semantic_skills
)

In [18]:
(df[["Job Description", "semantic_skills"]].head(10))

,Job Description,semantic_skills
0,Social Media Managers oversee an organizations...,"Leadership, Time Management, Social Media Mark..."
1,Frontend Web Developers design and implement u...,"Web Development, Frontend, Backend, JavaScript..."
2,Quality Control Managers establish and enforce...,"Project Management, Leadership, Compliance, Ri..."
3,"Wireless Network Engineers design, implement, ...","Networking, UI/UX Design, Mobile Development, ..."
4,A Conference Manager coordinates and manages c...,"Project Management, Leadership, Time Managemen..."
5,A Quality Assurance Analyst tests software and...,"Data Analysis, Agile, Testing, Auditing, Busin..."
6,A Classroom Teacher educates students in a spe...,Coaching
7,User Interface Designers focus on the visual a...,"Web Development, HTML, CSS, Data Visualization..."
8,Interaction Designers specialize in designing ...,"CSS, Communication, Operating Systems, System ..."
9,A Wedding Consultant assists couples in planni...,


In [19]:
df.to_csv(
    "/content/semantic_skill_extraction.csv",
    index=False
)